# NeoRacer Async Core Test

This notebook walks through every sensor, control and driver feature exposed by the student library racecar-neo-library at its tag `v0.9.0-neobotics`, running on the NeoRacer driver 0.9.0. Run each cell in order. The car must be powered on. The teleop stack starts at boot as the service `neoracer-teleop.service`; `racecar status` shows whether it is running. Do not start a second copy by hand.

Each section runs for a fixed window (10 s by default), shows live values and records its checks. The final "Summary" cell prints every check: PASS or FAIL for what a healthy car must do at rest, and YES or NO for what depends on what you did during the window (moved a stick, flipped the switch, turned object detection on, loaded a map).

**Section 6 drives the car.** Put the car on a stand with the wheels off the ground, or keep the handset's mode switch up until that section says to flip it down.

## 1. Initialize Racecar

In [ ]:
import sys, os, time, io

import numpy as np
import cv2 as cv
from IPython.display import display
import ipywidgets as widgets

import racecar_core
import racecar_utils as rc_utils

In [ ]:
def to_jpeg_bytes(bgr_image, resize=(320, 240)):
    """Convert a BGR numpy image to JPEG bytes for widget display."""
    if resize:
        bgr_image = cv.resize(bgr_image, resize)
    _, buf = cv.imencode('.jpg', bgr_image, [cv.IMWRITE_JPEG_QUALITY, 70])
    return buf.tobytes()

In [ ]:
# Create the racecar in real mode and start the async ROS2 executor.
# On the NeoRacer go_async() also waits up to 5 s for the first lidar scan.
rc = racecar_core.create_racecar(isSimulation=False)
rc.go_async()
print('Racecar initialized, async executor running.')
print('Waiting 3 seconds for topics to connect...')
time.sleep(3)

# Every section records its results here. The Summary cell prints them.
checks = {}   # must pass on a healthy car at rest
info = {}     # depends on what you did during the window

## 2. Forward Camera Stream (10 seconds)

`rc.camera` subscribes to `/camera/color` (`sensor_msgs/Image`) on the first read, and that first read can wait up to a second for a frame. The NeoRacer camera node publishes the USB webcam's JPEG frames with the encoding `jpeg`, and the library decodes them into 640 by 480 BGR images. There is no depth image on this car: `rc.camera.get_depth_image()` raises `NotImplementedError` (section 15 checks that).

In [ ]:
DURATION = 10
img_widget = widgets.Image(format='jpeg', width=320, height=240)
label = widgets.Label(value='Starting...')
display(widgets.VBox([label, img_widget]))

frame_count = 0
t_start = time.monotonic()

while time.monotonic() - t_start < DURATION:
    color_image = rc.camera.get_color_image_async()
    if color_image is not None:
        frame_count += 1
        img_widget.value = to_jpeg_bytes(color_image)
        elapsed = time.monotonic() - t_start
        fps = frame_count / elapsed if elapsed > 0 else 0
        label.value = f'Forward camera | Frame {frame_count} | {elapsed:.1f}s / {DURATION}s | {fps:.1f} FPS'
    time.sleep(0.05)

elapsed = time.monotonic() - t_start
rate = frame_count / elapsed if elapsed > 0 else 0
label.value = f'Forward camera complete: {frame_count} frames in {elapsed:.1f}s = {rate:.1f} FPS'

image = rc.camera.get_color_image_async()
checks['Camera: frames received (/camera/color)'] = frame_count > 0
checks['Camera: 640x480 color image'] = (
    image is not None and image.shape == (480, 640, 3)
    and rc.camera.get_width() == 640 and rc.camera.get_height() == 480
)

## 3. Lidar Stream (10 seconds)

`rc.lidar` subscribes to `/scan` (`sensor_msgs/LaserScan`). On the NeoRacer `rc.lidar.get_num_samples()` returns 1440: the LakiBeam1 lidar gives one sample every 0.25 degrees, in cm. Index 0 is straight ahead and the index grows clockwise, so index 360 is the right side and index 1080 the left. The lidar sees 270 degrees, so the rear 90 degrees (about indices 540 to 900) have no return and read 0. Always use `get_num_samples()` instead of hard-coding a length.

In [ ]:
DURATION = 10
MAX_RANGE_CM = 1000
img_widget = widgets.Image(format='jpeg', width=320, height=320)
label = widgets.Label(value='Starting...')
display(widgets.VBox([label, img_widget]))

frame_count = 0
t_start = time.monotonic()

while time.monotonic() - t_start < DURATION:
    scan = rc.lidar.get_samples_async()
    if scan is not None and len(scan) > 0:
        frame_count += 1
        radius = 160
        image = np.zeros((2 * radius, 2 * radius, 3), np.uint8)
        n = len(scan)
        for i in range(n):
            d = scan[i]
            if 0 < d < MAX_RANGE_CM:
                angle = 2 * np.pi * i / n
                length = radius * d / MAX_RANGE_CM
                r = int(radius - length * np.cos(angle))
                c = int(radius + length * np.sin(angle))
                if 0 <= r < 2 * radius and 0 <= c < 2 * radius:
                    image[r, c, 2] = 255
        cv.circle(image, (radius, radius), 3, (0, 255, 0), -1)
        img_widget.value = to_jpeg_bytes(image, resize=None)
        forward = rc_utils.get_lidar_average_distance(scan, 0)
        elapsed = time.monotonic() - t_start
        fps = frame_count / elapsed if elapsed > 0 else 0
        label.value = (
            f'Lidar | Samples: {n} | Frame {frame_count} | {elapsed:.1f}s / {DURATION}s | '
            f'{fps:.1f} Hz | Forward: {forward:.0f} cm'
        )
    time.sleep(0.05)

elapsed = time.monotonic() - t_start
rate = frame_count / elapsed if elapsed > 0 else 0
label.value = f'Lidar complete: {frame_count} scans in {elapsed:.1f}s = {rate:.1f} Hz'

scan = rc.lidar.get_samples_async()
checks['Lidar: scans received (/scan)'] = frame_count > 0
checks['Lidar: 1440 samples'] = (
    scan is not None and len(scan) == 1440 and rc.lidar.get_num_samples() == 1440
)

## 4. IMU / Physics Data

`rc.physics` reads `/imu/fused` (`sensor_msgs/Imu`) and `/mag` (`sensor_msgs/MagneticField`). On the NeoRacer the IMU is the QMI8658A on the OSCORE board, which fuses it and sends it to the Jetson. The axes follow REP-103: x forward, y left, z up. With the car level and still, the acceleration reads about +9.81 m/s² along z. Keep the car still during this cell.

In [ ]:
for i in range(5):
    accel = rc.physics.get_linear_acceleration()
    gyro = rc.physics.get_angular_velocity()
    mag = rc.physics.get_magnetic_field()

    print(f'--- Sample {i+1} ---')
    print(f'  Accel (x fwd, y left, z up): ({accel[0]:7.2f}, {accel[1]:7.2f}, {accel[2]:7.2f}) m/s^2')
    print(f'  Gyro (roll, pitch, yaw):     ({gyro[0]:7.3f}, {gyro[1]:7.3f}, {gyro[2]:7.3f}) rad/s')
    print(f'  Mag:                         ({mag[0]:+.3e}, {mag[1]:+.3e}, {mag[2]:+.3e}) T')
    time.sleep(0.2)

accel_mag = float(np.linalg.norm(accel))
print(f'\nAccel magnitude: {accel_mag:.2f} m/s^2 (expected ~9.81 when stationary)')
checks['IMU: acceleration magnitude in [8, 12] m/s^2 at rest'] = 8.0 < accel_mag < 12.0
checks['IMU: z axis up (accel z > 7 m/s^2 at rest)'] = float(accel[2]) > 7.0
if 8.0 < accel_mag < 12.0:
    print('PASS: IMU acceleration magnitude looks correct')
else:
    print('WARN: acceleration magnitude outside the expected range. Check the IMU calibration')

## 5. Object Detection (10 seconds)

`rc.vision` reads `/edgetpu/inference` (`vision_msgs/Detection2DArray`). The NeoRacer publishes it from a YOLO model on the Jetson's GPU; the topic keeps the name the library expects. Object detection is off at every boot. Turn it on from the car's web page on port 8080, or launch the stack with `inference_enable:=true`. `racecar setup ml` installs the model's Python packages; without them the node exits at startup.

The cell overlays the detection boxes on the live camera frame. With detection off, the frame shows and the detection count stays 0.

In [ ]:
DURATION = 10
img_widget = widgets.Image(format='jpeg', width=640, height=480)
label = widgets.Label(value='Starting...')
display(widgets.VBox([label, img_widget]))

frame_count = 0
total_detections = 0
t_start = time.monotonic()

while time.monotonic() - t_start < DURATION:
    color_image = rc.camera.get_color_image_async()
    detections = rc.vision.get_detections_async()
    if color_image is not None:
        frame_count += 1
        annotated = color_image.copy()
        for det in detections:
            cx, cy, w, h = det.bbox
            x1, y1 = int(cx - w / 2), int(cy - h / 2)
            x2, y2 = int(cx + w / 2), int(cy + h / 2)
            cv.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv.putText(
                annotated, f'{det.class_id} {det.score:.0%}', (x1, y1 - 8),
                cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2,
            )
        total_detections += len(detections)
        img_widget.value = to_jpeg_bytes(annotated, resize=None)
        elapsed = time.monotonic() - t_start
        fps = frame_count / elapsed if elapsed > 0 else 0
        label.value = (
            f'Detection | Frame {frame_count} | {elapsed:.1f}s / {DURATION}s | '
            f'{fps:.1f} FPS | Detections: {len(detections)} | Total: {total_detections}'
        )
    time.sleep(0.05)

elapsed = time.monotonic() - t_start
rate = frame_count / elapsed if elapsed > 0 else 0
label.value = f'Object detection complete: {frame_count} frames in {elapsed:.1f}s = {rate:.1f} FPS | Total detections: {total_detections}'

checks['Vision: get_detections_async() returns a list'] = isinstance(detections, list)
info['Vision: detections seen (needs object detection on)'] = total_detections > 0

## 6. Drive: who drives the car (30 seconds)

**This cell drives the car.** Put the car on a stand with the wheels off the ground, or keep the mode switch up until you are ready.

`rc.drive` publishes `/drive` (`ackermann_msgs/AckermannDriveStamped`). The driver's mux forwards every fresh `/drive` message to `/mux_out` and on to the OSCORE board, and the board selects who drives from the handset's mode switch:

- **Switch up or middle:** the handset drives. `rc.controller.Button.LB` reads as down. The program's commands reach `/mux_out`, but the board does not act on them.
- **Switch down:** the program drives. `rc.controller.Button.RB` reads as down for as long as the switch stays down. It is a switch, not a held bumper, so `is_down(RB)` stays True until you flip it back up.
- **Handset off:** after about 200 ms the board follows the program.

The board holds the wheel speed in m/s, so there is no ESC deadband to overcome: 1.0 on `/drive` is 6 m/s and 0.05 on `/drive` is about 0.3 m/s. While the switch is down, this cell sends a slow square wave at about 0.3 m/s and shows what reaches `/mux_out`.

In [ ]:
import rclpy as ros2
from ackermann_msgs.msg import AckermannDriveStamped
from rclpy.qos import QoSProfile, QoSReliabilityPolicy, QoSDurabilityPolicy

mux_node = ros2.create_node('mux_test_sub')
mux_qos = QoSProfile(depth=1)
mux_qos.reliability = QoSReliabilityPolicy.BEST_EFFORT
mux_qos.durability = QoSDurabilityPolicy.VOLATILE
mux_latest = [AckermannDriveStamped()]
mux_count = [0]

def _mux_cb(msg):
    mux_latest[0] = msg
    mux_count[0] += 1

mux_node.create_subscription(AckermannDriveStamped, '/mux_out', _mux_cb, mux_qos)
rc._RacecarReal__executor.add_node(mux_node)

DURATION = 30

# Bench values. What reaches /drive is command * STUDENT_MAX_SPEED, and 1.0 on /drive
# is 6 m/s on the NeoRacer, so 0.1 * 0.5 = 0.05 on /drive is about 0.3 m/s. The OSCORE
# board holds the wheel speed in m/s, so even this small value turns the wheels.
STUDENT_MAX_SPEED = 0.5
STUDENT_AMPLITUDE = 0.1
STUDENT_STEERING  = 0.4

timer_label = widgets.Label(value='Starting...')
switch_label = widgets.Label(value='Mode switch: ...')
mode_label = widgets.Label(value='Who drives: ...')
output_label = widgets.Label(value='Mux output: ...')
student_label = widgets.Label(value='Student /drive: ...')

display(widgets.VBox([timer_label, switch_label, mode_label, student_label, output_label]))

rc.drive.set_max_speed(STUDENT_MAX_SPEED)
rc.drive.stop()

drove_frames = 0
mux_peak = 0.0
t_start = time.monotonic()
while time.monotonic() - t_start < DURATION:
    elapsed = time.monotonic() - t_start
    timer_label.value = f'Drive | {elapsed:.1f}s / {DURATION}s'

    lb = rc.controller.is_down(rc.controller.Button.LB)
    rb = rc.controller.is_down(rc.controller.Button.RB)
    switch_label.value = f'Mode switch: {"DOWN (RB)" if rb else "UP or MIDDLE (LB)" if lb else "no reading"}'
    m = mux_latest[0]

    if rb and not lb:
        phase = (elapsed % 3.0) < 1.5
        speed = STUDENT_AMPLITUDE if phase else -STUDENT_AMPLITUDE
        angle = STUDENT_STEERING  if phase else -STUDENT_STEERING
        rc.drive.set_speed_angle(speed, angle)
        drove_frames += 1
        mux_peak = max(mux_peak, abs(m.drive.speed))
        mode_label.value = 'Who drives: the program. /drive is forwarded and the board follows it'
        published_speed = speed * STUDENT_MAX_SPEED
        student_label.value = (
            f'Student /drive: raw=({speed:+.2f}, {angle:+.2f})  '
            f'after max_speed={STUDENT_MAX_SPEED}: speed={published_speed:+.3f} (about {published_speed * 6:+.2f} m/s)'
        )
    elif lb and not rb:
        rc.drive.stop()
        mode_label.value = 'Who drives: the handset. /drive is forwarded but the board ignores it'
        student_label.value = 'Student /drive: stopped'
    else:
        rc.drive.stop()
        mode_label.value = 'Who drives: no mode switch reading (handset off or a failsafe frame)'
        student_label.value = 'Student /drive: stopped'

    output_label.value = (
        f'Mux out:   speed={m.drive.speed:+.3f}  steering_angle={m.drive.steering_angle:+.3f}  '
        f'messages={mux_count[0]}'
    )
    time.sleep(0.05)

rc.drive.stop()
timer_label.value = f'Drive test complete ({DURATION}s)'
rc._RacecarReal__executor.remove_node(mux_node)
mux_node.destroy_node()

checks['Drive: /mux_out received'] = mux_count[0] > 0
info['Drive: the program drove (switch was down)'] = drove_frames > 0
info['Drive: /mux_out carried the program speed while the switch was down'] = mux_peak > 0.01

## 7. Battery Voltage

`rc.physics.get_battery_voltage()` returns the pack voltage in volts, read from `/battery/state` (`sensor_msgs/BatteryState`), which the controller node publishes about every two seconds. Expect roughly four samples in the window below. A reading of 0 means none has arrived yet.

`rc.physics.get_battery_current()` raises `NotImplementedError`: the NeoRacer's OSCORE board has no current sensor (section 15 checks that).

In [ ]:
DURATION = 8

timer_label = widgets.Label(value='Starting...')
volt_label = widgets.Label(value='Voltage: ...')
display(widgets.VBox([timer_label, volt_label]))

v_min, v_max = 1e9, -1e9
t_start = time.monotonic()
while time.monotonic() - t_start < DURATION:
    v = rc.physics.get_battery_voltage()
    v_min, v_max = min(v_min, v), max(v_max, v)
    elapsed = time.monotonic() - t_start
    timer_label.value = f'Battery | {elapsed:.1f}s / {DURATION}s'
    volt_label.value = f'Voltage: {v:6.2f} V   (min {v_min:.2f}, max {v_max:.2f})'
    time.sleep(0.1)

v = rc.physics.get_battery_voltage()
power_ok = 9.0 < v < 13.0
checks['Battery: voltage in [9, 13] V (/battery/state)'] = power_ok
print(f'Final: {v:.2f} V')
print('PASS: voltage in [9, 13] V' if power_ok
      else 'WARN: voltage outside [9, 13] V. A reading of 0 means no /battery/state message has arrived yet')

## 8. Handset Channels

`rc.physics.get_rc_channels()` returns the first eight channels of the handset's receiver, each normalized to [-1, 1], from `/rc/channels`. Turn the handset on and move both sticks and the mode switch. The driver uses four channels: 0 is the steering stick (x), 2 is the throttle stick (y), 6 is the mode switch and 7 is the speed switch. A switch reads -1, 0 or +1. The other channels are not read by the driver.

In [ ]:
DURATION = 10
CH_NAMES = ['0 steering stick X', '1 (not used)', '2 throttle stick Y', '3 (not used)',
            '4 (not used)', '5 (not used)', '6 mode switch', '7 speed switch']

timer_label = widgets.Label(value='Starting... (turn the handset ON, move the sticks and the mode switch)')
ch_labels = [widgets.Label(value=f'{n}: 0.00') for n in CH_NAMES]
display(widgets.VBox([timer_label] + ch_labels))

mins = [9.0] * 8
maxs = [-9.0] * 8
t_start = time.monotonic()
while time.monotonic() - t_start < DURATION:
    ch = rc.physics.get_rc_channels()               # 8 values, each normalized to [-1, 1]
    for i in range(8):
        v = float(ch[i])
        mins[i] = min(mins[i], v)
        maxs[i] = max(maxs[i], v)
        ch_labels[i].value = f'{CH_NAMES[i]}: {v:+.2f}   (min {mins[i]:+.2f}, max {maxs[i]:+.2f})'
    elapsed = time.monotonic() - t_start
    timer_label.value = f'Handset channels | {elapsed:.1f}s / {DURATION}s'
    time.sleep(0.05)

moved = sum(1 for i in range(8) if (maxs[i] - mins[i]) > 0.3)
checks['Handset: 8 channels (/rc/channels)'] = len(ch) == 8
info['Handset: a channel moved'] = moved > 0
print(f'{moved} channel(s) swung more than 0.3 during the window.')
print('PASS: handset detected and channels responding' if moved > 0
      else 'WARN: no channel moved. Is the handset on and bound?')

## 9. Handset through rc.controller (10 seconds)

Labs read the handset through `rc.controller`, which the driver fills from `/joy`. On the NeoRacer:

- `Joystick.LEFT` y is the throttle stick, +1 fully forward. Its x is always 0.
- `Joystick.RIGHT` x is the steering stick, +1 fully right. Its y is always 0.
- `Button.RB` is down while the mode switch is down. `Button.LB` is down while it is up or in the middle.
- The triggers always read 0, and the A, B, X, Y and stick-click buttons are never pressed.

Move both sticks and flip the mode switch during the window.

In [ ]:
DURATION = 10
B = rc.controller.Button
J = rc.controller.Joystick
T = rc.controller.Trigger

timer_label = widgets.Label(value='Starting... (move the sticks and flip the mode switch)')
stick_label = widgets.Label(value='Sticks: ...')
switch_label = widgets.Label(value='Mode switch: ...')
display(widgets.VBox([timer_label, stick_label, switch_label]))

throttle_peak = 0.0
steering_peak = 0.0
trigger_peak = 0.0
saw_rb = saw_lb = False
t_start = time.monotonic()
while time.monotonic() - t_start < DURATION:
    lx, ly = rc.controller.get_joystick(J.LEFT)
    rx, ry = rc.controller.get_joystick(J.RIGHT)
    rb = rc.controller.is_down(B.RB)
    lb = rc.controller.is_down(B.LB)
    throttle_peak = max(throttle_peak, abs(ly))
    steering_peak = max(steering_peak, abs(rx))
    trigger_peak = max(trigger_peak, rc.controller.get_trigger(T.LEFT), rc.controller.get_trigger(T.RIGHT))
    saw_rb = saw_rb or rb
    saw_lb = saw_lb or lb
    elapsed = time.monotonic() - t_start
    timer_label.value = f'Handset | {elapsed:.1f}s / {DURATION}s'
    stick_label.value = (
        f'Throttle (LEFT y): {ly:+.2f}   Steering (RIGHT x): {rx:+.2f}   '
        f'(LEFT x {lx:+.2f} and RIGHT y {ry:+.2f} stay 0)'
    )
    switch_label.value = f'Mode switch: {"DOWN (RB)" if rb else "UP or MIDDLE (LB)" if lb else "no reading"}'
    time.sleep(0.05)

checks['Controller: triggers read 0 (the handset has none)'] = trigger_peak == 0.0
info['Controller: throttle stick moved (Joystick.LEFT y)'] = throttle_peak > 0.3
info['Controller: steering stick moved (Joystick.RIGHT x)'] = steering_peak > 0.3
info['Controller: mode switch seen down (Button.RB)'] = saw_rb
info['Controller: mode switch seen up or middle (Button.LB)'] = saw_lb
print(f'Throttle peak {throttle_peak:.2f}, steering peak {steering_peak:.2f}, RB seen {saw_rb}, LB seen {saw_lb}')

## 10. Encoder Speed

`rc.physics.get_encoder_speed()` returns the car's forward speed in m/s from `/encoder/speed`, which the OSCORE board reports from the motor encoder. The library does not expose raw motor rpm. Spin a wheel by hand, or drive the car with the handset, to see it change.

In [ ]:
DURATION = 10

timer_label = widgets.Label(value='Starting... (spin a wheel by hand or drive with the handset)')
speed_label = widgets.Label(value='Speed: ...')
display(widgets.VBox([timer_label, speed_label]))

max_abs = 0.0
t_start = time.monotonic()
while time.monotonic() - t_start < DURATION:
    speed = rc.physics.get_encoder_speed()          # m/s, signed
    max_abs = max(max_abs, abs(speed))
    elapsed = time.monotonic() - t_start
    timer_label.value = f'Encoder | {elapsed:.1f}s / {DURATION}s'
    speed_label.value = f'Speed: {speed:+.3f} m/s   (max |v| {max_abs:.3f})'
    time.sleep(0.05)

checks['Encoder: speed readable (/encoder/speed)'] = isinstance(speed, float)
info['Encoder: motion seen'] = max_abs > 0.0
print(f'Peak speed seen: {max_abs:.3f} m/s')
print('PASS: encoder responded to motion' if max_abs > 0.0
      else 'INFO: no motion detected. Spin a wheel during the window to confirm the encoder')

## 11. Dot Matrix Display

The NeoRacer panel is an 8x8 module that renders text and nothing else, so `/dotmatrix/text` (`std_msgs/String`) is the whole interface and `rc.display.show_text()` is the only call that reaches it. `set_matrix()`, `get_matrix()` and `set_matrix_intensity()` raise `NotImplementedError` (section 15 checks that), and there is no `/dotmatrix/pixels` on this car.

The panel holds the last frame written, so the section ends on `rc.display.clear()`, which returns it to the idle frame the driver writes at startup. A new frame is applied only when the current scroll cycle ends, so writes sent closer together than one cycle can be swallowed.

In [ ]:
# show_text() is the whole dot matrix interface on this car; the panel takes text only.
rc.display.show_text('NEORACER')
print('Sent "NEORACER" to /dotmatrix/text. Holding 5 s...')
time.sleep(5)

# Anything wider than the panel scrolls automatically.
rc.display.show_text('CHECK OK')
print('Sent a longer string; the panel scrolls it.')
time.sleep(10)

# One write to end on; a second before the scroll cycle ends can be swallowed.
rc.display.clear()
checks['Dot matrix: show_text() and clear() accepted (/dotmatrix/text)'] = True
print('Cleared to the idle frame.')

## 12. Telemetry: record and visualize

`rc.telemetry.declare_variables(...)` opens a timestamped CSV under `labs/logs/`, the labs folder next to the library (`~/jupyter_ws/neoracer-os/labs/logs/` on a car); `record(...)` appends a row; `visualize()` writes a sibling PNG. We log the acceleration magnitude and a synthetic sine for 5 seconds and then plot.

In [ ]:
rc.telemetry.declare_variables('accel_mag', 'sine')

DURATION = 5
t_start = time.monotonic()
n = 0
while time.monotonic() - t_start < DURATION:
    t = time.monotonic() - t_start
    a = rc.physics.get_linear_acceleration()
    rc.telemetry.record(float(np.linalg.norm(a)), float(np.sin(2 * np.pi * 0.5 * t)))
    n += 1
    time.sleep(0.05)

rc.telemetry.visualize()
print(f'Logged {n} telemetry samples in {DURATION}s.')
print(f'CSV: {rc.telemetry._LOG_FILE_NAME}')
print(f'PNG: {rc.telemetry._PLOT_FILE_NAME}')
checks['Telemetry: CSV written'] = os.path.isfile(rc.telemetry._LOG_FILE_NAME)
checks['Telemetry: PNG written'] = os.path.isfile(rc.telemetry._PLOT_FILE_NAME)

## 13. Diagnostics

`/diagnostics` (`diagnostic_msgs/DiagnosticArray`) carries inference timing, frame counts and device health. On the NeoRacer only the driver's inference node publishes it, under the status name "YOLO Inference". `rc.telemetry.get_diagnostics()` returns a dict keyed by status name. Expect it to be empty while object detection is off.

In [ ]:
diag = rc.telemetry.get_diagnostics()

if diag:
    for name, entry in diag.items():
        level = entry['level']
        if isinstance(level, (bytes, bytearray)):
            level = int.from_bytes(level, byteorder='big')
        level_str = {0: 'OK', 1: 'WARN', 2: 'ERROR', 3: 'STALE'}.get(level, f'LVL:{level}')
        print(f'[{level_str}] {name}: {entry["message"]}')
        for k, v in entry.items():
            if k not in ('level', 'message', 'hardware_id'):
                print(f'    {k}: {v}')
        print()
else:
    print('No diagnostics received. Object detection is off at boot: turn it on from the car\'s web page on port 8080, or launch with inference_enable:=true.')

info['Diagnostics: received (needs object detection on)'] = bool(diag)

## 14. SLAM: map and pose (20 seconds)

`rc.slam` reads the car's position on a lidar map from the driver's autonomy service. Start mapping or load a saved map first, in the map viewer (`http://<car>:8088`) or with `racecar map start NAME` / `racecar map load NAME`. The cell draws the map (walls dark, free space light, unknown gray), the car (green when localized, orange when not), and its path. Drive the car with the handset (mode switch up) while it runs.

- `rc.slam.get_mode()`: `"mapping"`, `"localization"`, or `"off"`
- `rc.slam.get_pose_async()`: `(x, y, heading)` in meters and degrees, counterclockwise from the map's x axis
- `rc.slam.is_localized()`: whether the pose can be trusted; false within seconds of the car being lifted
- `rc.slam.get_map()`: an `OccupancyMap`; `data` row 0 is the bottom row (-1 unknown, 0 free, 100 wall)

In [ ]:
DURATION = 20
SCALE = 3  # screen pixels per map cell
img_widget = widgets.Image(format='jpeg')
label = widgets.Label(value='Starting...')
display(widgets.VBox([label, img_widget]))


def map_image(m):
    """BGR image of an OccupancyMap cropped to its known cells, top row first."""
    known = np.argwhere(m.data != -1)
    if len(known) == 0:
        return None, None
    (r0, c0), (r1, c1) = known.min(0), known.max(0) + 1
    shade = np.full((r1 - r0, c1 - c0), 170, np.uint8)
    block = m.data[r0:r1, c0:c1]
    shade[block == 0] = 245
    shade[block == 100] = 30
    image = cv.cvtColor(np.flipud(shade), cv.COLOR_GRAY2BGR)
    image = cv.resize(image, None, fx=SCALE, fy=SCALE, interpolation=cv.INTER_NEAREST)
    return image, (r0, c0, r1)


def to_pixel(m, crop, x, y):
    r0, c0, r1 = crop
    col = (x - m.origin[0]) / m.resolution - c0
    row = (y - m.origin[1]) / m.resolution
    return int(col * SCALE), int((r1 - row) * SCALE)


base, crop, shown_map = None, None, None
path = []
t_start = time.monotonic()
while time.monotonic() - t_start < DURATION:
    mode = rc.slam.get_mode()
    elapsed = time.monotonic() - t_start
    if mode == 'off':
        label.value = f'{elapsed:.0f}s: autonomy is off. Start mapping or load a map in the map viewer (port 8088).'
        time.sleep(0.5)
        continue
    m = rc.slam.get_map()
    if m is not None and m is not shown_map:
        base, crop = map_image(m)
        shown_map = m
    pose = rc.slam.get_pose_async()
    if base is None or pose is None:
        label.value = f'{elapsed:.0f}s: {mode}, waiting for the map and pose...'
        time.sleep(0.2)
        continue
    x, y, heading = pose
    path.append((x, y))
    image = base.copy()
    points = [to_pixel(m, crop, px, py) for px, py in path[-600:]]
    for a, b in zip(points, points[1:]):
        cv.line(image, a, b, (0, 140, 255), 1)
    tip = (x + 0.4 * np.cos(np.radians(heading)), y + 0.4 * np.sin(np.radians(heading)))
    color = (0, 180, 0) if rc.slam.is_localized() else (0, 140, 255)
    cv.arrowedLine(image, to_pixel(m, crop, x, y), to_pixel(m, crop, *tip), color, 2, tipLength=0.4)
    img_widget.value = to_jpeg_bytes(image, resize=None)
    label.value = (
        f'{elapsed:.0f}s / {DURATION}s | {mode} on {rc.slam.get_map_name()} | '
        f'localized: {rc.slam.is_localized()} | x {x:.2f} m, y {y:.2f} m, heading {heading:.0f} deg | {m}'
    )
    time.sleep(0.1)

label.value += ' | done'
mode = rc.slam.get_mode()
checks['SLAM: get_mode() returns a known value'] = mode in ('mapping', 'localization', 'off')
info['SLAM: pose received (needs mapping or a loaded map)'] = len(path) > 0

## 15. Not available on the NeoRacer

These calls exist in the library for hardware the NeoRacer does not have. Each raises `NotImplementedError` with a message naming the missing part. That is the expected behavior, so the cell checks that they raise.

In [ ]:
unsupported = {
    'Depth image: rc.camera.get_depth_image()': lambda: rc.camera.get_depth_image(),
    'Battery current: rc.physics.get_battery_current()': lambda: rc.physics.get_battery_current(),
    'LED strip: rc.led.get_num_pixels()': lambda: rc.led.get_num_pixels(),
    'LED strip: rc.led.set_color()': lambda: rc.led.set_color((255, 0, 0)),
    'Dot matrix pixels: rc.display.get_matrix()': lambda: rc.display.get_matrix(),
    'Dot matrix pixels: rc.display.set_matrix()': lambda: rc.display.set_matrix(rc.display.new_matrix()),
    'Dot matrix pixels: rc.display.set_matrix_intensity()': lambda: rc.display.set_matrix_intensity(0.5),
}
for name, call in unsupported.items():
    try:
        call()
        checks[f'Unsupported: {name} raises NotImplementedError'] = False
        print(f'UNEXPECTED: {name} did not raise')
    except NotImplementedError as e:
        checks[f'Unsupported: {name} raises NotImplementedError'] = True
        print(f'expected: {name}\n    {e}')

## 16. Summary

In [ ]:
checks['Utils: rc_utils.get_data_dir() exists'] = os.path.isdir(rc_utils.get_data_dir())

failed = [name for name, ok in checks.items() if not ok]
overall = 'ALL PASS' if not failed else f'{len(failed)} FAILED, see above'

print('=' * 72)
print('  NeoRacer Async Core Test Results')
print('=' * 72)
for name, ok in checks.items():
    print(f'  [{"PASS" if ok else "FAIL"}] {name}')
print('-' * 72)
print('  Depends on what happened during the windows:')
for name, ok in info.items():
    print(f'  [{"YES " if ok else "NO  "}] {name}')
print('=' * 72)
print(f'  Overall: {overall}')
print('=' * 72)